# Phase 2 — LHS Sampling + SWMM Automation Pipeline

Builds a training dataset by running the base SWMM model 500 times under Latin Hypercube-sampled combinations of 5 uncertain parameters.

**Flow:** LHS sample -> edit `base_model.inp` -> run `runswmm.exe` -> parse `.rpt` -> `dataset.csv`.



## 1. Config & paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- Locate repo root (works whether launched from repo root or /notebooks) ---
# Open this notebook from the project folder or the notebooks folder.
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
assert (PROJECT / "notebooks").is_dir(), "Open from the project or notebooks folder."
# --- Paths ---
# SWMM_EXE is machine-specific: set this to YOUR SWMM install (see README).
SWMM_EXE = Path(r"D:\EPA SWMM 5.2.4 (64-bit)\runswmm.exe")
BASE_INP = PROJECT / "notebooks" / "base_model.inp"
RUNS_DIR = PROJECT / "runs"
DATA_DIR = PROJECT / "output"
RUNS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# --- Experiment settings ---
N_RUNS = 500
SEED = 42

# --- Uncertain parameters: name -> (low, high) ---
PARAM_RANGES = {
    "rain_mult":   (0.7, 1.5),      # multiply the 2-yr rainfall timeseries
    "imperv_mult": (0.7, 1.3),      # multiply each subcatchment %Imperv (clip 0-100)
    "surf_n":      (0.011, 0.030),  # impervious overland Manning n (absolute)
    "infil_min":   (0.05, 1.5),     # Horton MinRate, in/hr (absolute)
    "pipe_n":      (0.011, 0.030),  # conduit Manning roughness (absolute)
}
PARAM_NAMES = list(PARAM_RANGES)

assert SWMM_EXE.exists(), f"runswmm.exe not found at {SWMM_EXE}"
assert BASE_INP.exists(), f"base_model.inp not found at {BASE_INP}"
print("Config OK. Base model:", BASE_INP.name, "| runs dir:", RUNS_DIR)

## 2. Latin Hypercube sampling

`scipy.stats.qmc.LatinHypercube` draws 500 points in the 5-D unit cube, then we scale each column to its physical range. A fixed `SEED` makes this reproducible.

In [ ]:
from scipy.stats import qmc

sampler = qmc.LatinHypercube(d=len(PARAM_RANGES), seed=SEED)
unit = sampler.random(n=N_RUNS)              # (N_RUNS, 5) in [0, 1)
lows = np.array([lo for lo, hi in PARAM_RANGES.values()])
highs = np.array([hi for lo, hi in PARAM_RANGES.values()])
scaled = qmc.scale(unit, lows, highs)

samples = pd.DataFrame(scaled, columns=PARAM_NAMES)
samples.index.name = "run_id"

## 3. Edit the `.inp` per sample

Read each line and keep track of the current section. Split the data into columns, change the sampled value, and join the columns again. Keep blank lines, comments, and other sections as they are. The check below shows the changes for sample 0.


In [ ]:
def write_scenario(base_text, params):
    """Change the five sampled inputs in the base INP text."""
    out = []
    current = ""

    for line in base_text.splitlines():
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            current = stripped.upper()
            out.append(line)
            continue
        if not stripped or stripped.startswith(";"):
            out.append(line)
            continue

        data = line.split()
        if current == "[TIMESERIES]" and data[0] == "2-yr":
            data[-1] = f"{float(data[-1]) * params['rain_mult']:g}"
        elif current == "[SUBCATCHMENTS]":
            # %Imperv must stay between 0 and 100.
            imperv = float(data[4]) * params["imperv_mult"]
            data[4] = f"{min(100, max(0, imperv)):g}"
        elif current == "[SUBAREAS]":
            data[1] = f"{params['surf_n']:g}"       # N-Imperv
        elif current == "[INFILTRATION]":
            data[2] = f"{params['infil_min']:g}"    # Horton minimum rate
        elif current == "[CONDUITS]":
            data[4] = f"{params['pipe_n']:g}"       # Manning roughness
        else:
            out.append(line)
            continue

        out.append("  ".join(data))

    return "\n".join(out) + "\n"


## 4. Run SWMM and read the `.rpt`

Find each report heading between its two rows of asterisks. Search within that section for Out1, flooding loss, and routing continuity error. Count the rows in the Node Flooding Summary; no flooding gives zero rows. Missing required results raise an error so an incomplete report cannot become an OK run.


In [ ]:
import subprocess

def run_swmm(inp_path):
    """Run a scenario through runswmm.exe. Returns (rpt_path, out_path, proc)."""
    inp_path = Path(inp_path)
    rpt_path = inp_path.with_suffix(".rpt")
    out_path = inp_path.with_suffix(".out")
    proc = subprocess.run(
        [str(SWMM_EXE), str(inp_path), str(rpt_path), str(out_path)],
        capture_output=True, text=True,
    )
    return rpt_path, out_path, proc

def parse_rpt(rpt_text):
    """Read the three targets and routing continuity error from the RPT."""
    lines = rpt_text.splitlines()
    headings = []
    for i in range(1, len(lines) - 1):
        if lines[i - 1].strip().startswith("***") and lines[i + 1].strip().startswith("***"):
            headings.append(i)

    def section_lines(heading):
        for n, start in enumerate(headings):
            if heading.upper() in lines[start].upper():
                end = len(lines)
                if n + 1 < len(headings):
                    end = headings[n + 1] - 1
                return lines[start + 2:end]
        return []

    def search_line(heading, item):
        for line in section_lines(heading):
            data = line.split()
            # Match a whole row label, so Out1 cannot match Out10.
            if data[:len(item.split())] == item.split():
                return data
        raise ValueError(f"Missing {item} in {heading}")

    outfall = search_line("Outfall Loading Summary", "Out1")
    flooding = search_line("Flow Routing Continuity", "Flooding Loss")
    continuity = search_line("Flow Routing Continuity", "Continuity Error (%)")

    num_flooded_nodes = 0
    separators = 0
    for line in section_lines("Node Flooding Summary"):
        stripped = line.strip()
        if stripped.startswith("---"):
            separators += 1
            continue
        if separators >= 2:
            if not stripped:
                break
            num_flooded_nodes += 1

    return {
        "peak_discharge_cfs": float(outfall[3]),       # Max Flow, CFS
        "total_flood_vol": float(flooding[-1]),       # Volume, 10^6 gal
        "num_flooded_nodes": num_flooded_nodes,
        "continuity_error_pct": float(continuity[-1]),
    }

# Run the first sample and read its report.
demo_inp = RUNS_DIR / "scenario_000.inp"
demo_inp.write_text(demo)
rpt, out, proc = run_swmm(demo_inp)
if proc.returncode != 0:
    raise RuntimeError(f"SWMM failed: {proc.stdout} {proc.stderr}")
print(parse_rpt(rpt.read_text()))


## 5. Batch loop (500 runs)

Each sample is written, run, and parsed. A `try/except` isolates failures so one bad run flags its row instead of crashing the batch. The binary `.out` is deleted after parsing; `.inp`/`.rpt` are kept for auditability.

In [ ]:
import time

rows = []
t0 = time.time()

for run_id, row in samples.iterrows():
    params = row.to_dict()
    inp_path = RUNS_DIR / f"scenario_{run_id:03d}.inp"
    rec = {"run_id": run_id, **params}
    try:
        inp_path.write_text(write_scenario(base_text, params))
        rpt_path, out_path, proc = run_swmm(inp_path)
        if proc.returncode != 0 or not rpt_path.exists():
            rec["status"] = "failed"
        else:
            rec.update(parse_rpt(rpt_path.read_text()))
            ce = rec["continuity_error_pct"]
            rec["status"] = "flagged" if abs(ce) > 5 else "ok"
        if out_path.exists():
            out_path.unlink()           # drop the binary .out to save space
    except Exception as e:
        rec["status"] = "failed"
        rec["error"] = str(e)
    rows.append(rec)
    if (run_id + 1) % 10 == 0:
        print(f"  {run_id + 1}/{N_RUNS} done  ({time.time() - t0:.1f}s)")

results = pd.DataFrame(rows).set_index("run_id")
print(f"\nBatch finished in {time.time() - t0:.1f}s")
print(results["status"].value_counts())

## 6. Assemble & save `dataset.csv`

Merge features + targets + QC columns, save, then sanity-check: how many runs flooded, the spread of each target, and how peak discharge responds to each input.

In [ ]:
import matplotlib.pyplot as plt

feature_cols = PARAM_NAMES
target_cols = ["peak_discharge_cfs", "total_flood_vol", "num_flooded_nodes"]
qc_cols = ["continuity_error_pct", "status"]

dataset = results[feature_cols + target_cols + qc_cols].copy()
dataset.to_csv(DATA_DIR / "dataset.csv")
print("Saved dataset.csv:", dataset.shape)
display(dataset.head())

# Sanity: how many runs flooded? Peak discharge spread?
ok = dataset[dataset["status"] != "failed"]
print(f"\nRuns OK: {len(ok)} | flooded (>0 vol): {(ok['total_flood_vol'] > 0).sum()}")
print(ok[target_cols].describe().round(3))

# Target distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, target_cols):
    ax.hist(ok[col].dropna(), bins=20, color="#3b7dd8", edgecolor="white")
    ax.set_title(col)
plt.tight_layout()
plt.show()

# Each feature vs peak discharge
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for ax, col in zip(axes, feature_cols):
    ax.scatter(ok[col], ok["peak_discharge_cfs"], s=14, alpha=0.7)
    ax.set_xlabel(col)
    ax.set_ylabel("peak CFS")
plt.tight_layout()
plt.show()

## 7. Learning demo — pyswmm (in-process)

Optional. Reads the *same* peak discharge for run 0 by stepping the simulation in-process, to contrast with the `.rpt` text parsing above. Install first if needed: `pip install pyswmm`.

In [7]:
# Learning demo: read the SAME peak discharge in-process with pyswmm,
# to compare against our .rpt text parsing.
#   If not installed:  pip install pyswmm
try:
    from pyswmm import Simulation, Nodes

    scen = RUNS_DIR / "scenario_000.inp"
    peak = 0.0
    with Simulation(str(scen)) as sim:
        out1 = Nodes(sim)["Out1"]
        for _ in sim:
            peak = max(peak, out1.total_inflow)

    print(f"pyswmm  peak inflow at Out1 : {peak:6.2f} CFS")
    print(f".rpt    parsed peak (run 0) : {results.loc[0, 'peak_discharge_cfs']:6.2f} CFS")
    print("These should match closely (both read the same simulation).")
except ImportError:
    print("pyswmm not installed. To try this learning cell, run:  pip install pyswmm")

pyswmm  peak inflow at Out1 :  22.98 CFS
.rpt    parsed peak (run 0) :  22.98 CFS
These should match closely (both read the same simulation).
